# **眼電訊號量測實作單元(一)：你剛剛是不是往右看了？是不是在偷偷想什麼？**

> 在這次的實驗中，鐵克將會帶大家初步用斜率公式分析EOG訊號。

> 並用動畫工具將EOG訊號分析出的結果做成動畫，讓大家對EOG訊號的強大之處有更深的感受！

# 0. 先收錄一些會用到的公式和函式吧！

In [1]:
#@title
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rc
from matplotlib import animation

#eyemove_range = 1
#time_speed = 1
#fig = plt.figure(figsize=(12,5))
def animation_parameter(eyemove_range, time_speed):
  eyemove_range = eyemove_range/60
  time_speed = 25/time_speed
  return eyemove_range, time_speed


def drawframe(n):
    if(n+1 > np.size(data4)):
      n = np.size(data4)-2
    x1 = np.arange(0,n+1)
    EOG.set_data(x1,data4[0:n+1])

    x = np.linspace(0, 2, 1000)
    y1 = np.sin(2 * np.pi * (x - 0.01 * n))
    y2 = 0.7*np.cos(2 * np.pi * (x - 0.01 * n))

    eye1.set_data(y1[0:500]+1.5,y2[0:500])
    eyeball1.set_data((data4[n]*eyemove_range)+1.5,0)
    
    eye2.set_data(y1[0:500]-1.5,y2[0:500])
    eyeball2.set_data((data4[n]*eyemove_range)-1.5,0)
    txt_title.set_text('Frame = {0:4d}'.format(n))

    return eye1,

def animation_produce(time_speed):
  
  # equivalent to rcParams['animation.html'] = 'html5'
  rc('animation', html='html5')


  # blit=True re-draws only the parts that have changed.
  show_animation = animation.FuncAnimation(fig, drawframe, frames=500, interval=time_speed, blit=True)

  return show_animation
def create_screen(y_length):
  # create a figure and axes
  fig = plt.figure(figsize=(12,5))
  ax1 = plt.subplot(1,2,1)   
  ax2 = plt.subplot(1,2,2)

  # set up the subplots as needed
  ax1.set_xlim(( 0, 500))            
  ax1.set_ylim((-(y_length/2), (y_length/2)))
  #ax1.set_ylim((-200, 200))
  ax1.set_xlabel('Time')
  ax1.set_ylabel('Magnitude')

  ax2.set_xlim((-4,4))
  ax2.set_ylim((-2,2))
  ax2.set_xlabel('X')
  ax2.set_ylabel('Y')
  ax2.set_title('Eyeball Move')

  # create objects that will change in the animation. These are
  # initially empty, and will be given new values for each frame
  # in the animation.
  txt_title = ax1.set_title('')
  EOG, = ax1.plot([], [], 'b', lw=2)     # ax.plot returns a list of 2D line objects


  eyeball1, = ax2.plot([], [], 'k.', ms=70)
  eye1, = ax2.plot([], [], 'k', lw=4)
  eyeball2, = ax2.plot([], [], 'k.', ms=70)
  eye2, = ax2.plot([], [], 'k', lw=4)
  ax1.legend(['EOG']);
  return fig,ax1,ax2,txt_title,EOG,eyeball1,eye1,eyeball2,eye2


# 1.先將自己左右看的眼電訊號丟進程式庫吧！

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
df = pd.read_csv('範例眼電訊號.txt') #若用自己的檔案，記得修改('xxxx')裡面的檔名唷！
df.info()
data = np.array(df)
data2 = data[0 : , 0]

# 2. 畫出自己尚未做任何處理的眼電訊號長什麼樣子吧！

In [ ]:
#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data2, #秀出你測到的眼電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="你測到的眼電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

# 3. 試著將EOG代入斜率公式看看效果如何吧！

In [5]:
def slopee(y1,y2):        #將斜率公式定義(def)成一個函式
    
    x = (y2 - y1) / 2 #Middle的取樣頻率是500 Hz, 點和點之間的時間間隔為2毫秒 
    return x


slopee_space = 30 #y2與y1的點數間隔 (大家可以練習將這個數值調大調小對波型的影響)

counter = slopee_space #由於是往前取點來套斜率公式，起始點也要往前，而不能是0
data3 = np.zeros(np.size(data2)) #事先建立好斜率算出來要存放的空間
while(counter < np.size(data2)):
  data3[counter] = slopee(data2[counter], data2[counter-slopee_space])  
  counter += 1

In [ ]:
#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data3, #秀出斜率公式轉換後的眼電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="斜率公式轉換後的眼電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

> 若y1, y2的間隔取的適當，上面的波形的物理意義即為眼球的動向(波形往上即為眼球往右)。


# 4. 試著將EOG結果呈現在動畫裡面吧！

> 由於EOG數據的取樣頻率較高(500Hz)，一般動畫呈現不需要這麼高的影片偵數(通常25~50 FPS即可)

> 我們試著重新修改數據，將原本數據每隔十點才取一次，得到新的取樣頻率較低的數據(50 FPS)。我們稱這個動作為「降低取樣」。

In [ ]:
resampler = 10 #降低取樣的倍率

data4 = np.zeros(int(np.size(data3)/resampler)) #事先建立降低取樣後的資料存放的位置

counter = 0 #用來定位降低取樣後的資料位置
counter_resample = 1 #用來計算每到resampler的數值便取樣一次的counter

for i in data3:
  if(counter_resample == resampler): #每loop到一次resampler的值就取樣一次
    data4[counter] = i
    counter += 1
    counter_resample = 0
  counter_resample += 1  

#稍微設定一下畫布資訊！ 
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data4, #秀出降低取樣後的眼電訊號吧！
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="降低取樣後的眼電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

> 利用上面降低取樣的數據，往下製作出一部眼球擺動的動畫吧！

In [ ]:
(fig,ax1,ax2,txt_title,EOG,eyeball1,eye1,eyeball2,eye2) = create_screen(200) #這個200是用來設定畫布的高度(y軸)，若你的數值較窄或較寬都可以調整這個數值

(eyemove_range, time_speed) = animation_parameter(1, 1) #(眼球左右擺動幅度, 動畫速度) 每個人的電訊號強度不一，透過微調參數來達到最好的影片效果吧！
show_animation = animation_produce(time_speed) #安排動畫設定

In [ ]:
show_animation #開始製作動畫！製作並輸出僅需1~2分鐘！

#結論
---

1.   眼電訊號的分析並非難事，認識眼電訊號並了解頭部除了腦波，還有不同的電訊號存在。
2.   降低取樣能幫助數據在不同應用達到最佳的呈現。
3.   動畫好可愛